# Heed Wake Word: train your own in Colab

Type a phrase, synthesize it across hundreds of voices, train a tiny model on a
free GPU, and download an ONNX / TFLite model you can run in the browser, on a
phone, or in Python. Nothing to install on your own machine.

**First:** switch the runtime to GPU (Runtime, Change runtime type, T4 GPU).
Training works on CPU too, just slower.

Project home: https://github.com/AndreiBulzan/heed-wakeword

## 1. Install

In [ ]:
# A GPU runtime (Runtime > Change runtime type > T4 GPU) makes TTS and training much faster.
!pip install -q "heed-wakeword[tts,export]"
# Optional: also emit a .tflite (Android NNAPI / iOS Core ML delegate paths).
# !pip install -q litert-torch
!heed --version

## 2. Pick your wake phrase

Edit `PHRASE`. A two-word phrase ("hey jasper", "ok ranch") triggers far less by
accident than a single common word. The other knobs are sensible defaults.

In [ ]:
PHRASE = "hey jasper"  #@param {type:"string"}
N_SEED_POSITIVES = 12  #@param {type:"integer"}
N_TTS_POSITIVES = 250  #@param {type:"integer"}
EPOCHS = 35  #@param {type:"integer"}

import re
PROJECT = re.sub(r"[^a-z0-9]+", "_", PHRASE.lower()).strip("_") or "wake"
print("phrase:", repr(PHRASE), "| project dir:", PROJECT)

## 3. Create the project and download the multi-speaker voice

The voice is LibriTTS-R medium (904 speakers, about 78 MB). Heed holds out a
fixed set of speakers so the trained model can be evaluated on voices it never
saw during training.

In [ ]:
!heed init {PROJECT} --phrase "{PHRASE}"
!heed download-tts

## 4. Synthesize seed clips

We seed the project with a handful of synthetic positives and a few phonetic
neighbor negatives ("hey", the rest of the phrase alone, similar-sounding
phrases). The training step below synthesizes many more speakers and a built-in
pool of hard negatives on top of these.

In [ ]:
from pathlib import Path
from heed.tts import synthesize_phrase, phonetic_neighbor_distractors
from heed.audio import save_wav

proj = Path(PROJECT)
pos = synthesize_phrase(PHRASE, N_SEED_POSITIVES, seed=1)
for i, clip in enumerate(pos):
    save_wav(proj / "positive" / f"pos_{i:03d}.wav", clip)

distractors = phonetic_neighbor_distractors(PHRASE, max_neighbors=12)
k = 0
for text in distractors:
    for clip in synthesize_phrase(text, 2, seed=100 + k):
        save_wav(proj / "negative" / f"neg_{k:03d}.wav", clip)
        k += 1

print(f"seed positives: {len(list((proj / 'positive').glob('*.wav')))}")
print(f"seed negatives: {k}  (distractors: {distractors[:6]} ...)")

## 5. Train

This synthesizes `N_TTS_POSITIVES` more speakers saying the phrase, adds a pool
of confusable phrases as hard negatives, augments everything (speaker warp,
reverb, noise, gain), and fits the model. The printed held-out evaluation
(TPR / FPR on speakers held out of training) is your honest signal for whether
the model generalizes beyond the voices it trained on.

Add `--kokoro-pos 150` for a second TTS family (cross-engine robustness) after
running the optional `!heed download-kokoro`.

In [ ]:
!heed train {PROJECT} --epochs {EPOCHS} --tts-pos {N_TTS_POSITIVES}

## 6. Export

Produces `wake.onnx` (float32), `wake.int8.onnx` (quantized), `wake.json` (the
preprocessing contract), and a deployment README. A `wake.tflite` is added too
if you installed `litert-torch` in step 1.

In [ ]:
!heed export {PROJECT}
!ls -la {PROJECT}/export

## 7. Download your model

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(f"{PROJECT}_model", "zip", f"{PROJECT}/export")
files.download(archive)

## What you got, and where to run it

The zip contains a sub-250 KB model plus its `wake.json` preprocessing contract.
It runs the same way everywhere:

- **Browser:** drop `wake.onnx` + `wake.json` into `examples/inference_browser/`
  and host the folder statically (it is fully client-side).
- **iOS / Android:** copy the files into a slot in
  `examples/inference_react_native/assets/`.
- **Python:** `onnxruntime` plus the snippet in the export README.

This model was trained purely on synthetic voices, so it is speaker-independent
by design. For the best accuracy on your own microphone and room, record a few
positives in your own voice and retrain. The local studio (`heed ui`) makes that
a two-minute job.